In [1]:
import numpy as np
from openpi_client import websocket_client_policy

client = websocket_client_policy.WebsocketClientPolicy(host="localhost", port=18000)
print("Server metadata:", client.get_server_metadata())

observation = {
    "observation/exterior_image_1_left": np.random.randint(256, size=(224, 224, 3), dtype=np.uint8),
    "observation/wrist_image_left": np.random.randint(256, size=(224, 224, 3), dtype=np.uint8),
    "observation/joint_position": np.random.rand(7),
    "observation/gripper_position": np.random.rand(1),
    "prompt": "pick up the cube and place it in the basket",
}

result = client.infer(observation)
actions = np.asarray(result["actions"])
print("Action chunk shape:", actions.shape)   # expect (15, 8)
print("First action:", actions[0])

Server metadata: {}
Action chunk shape: (15, 8)
First action: [ 0.00692769 -0.01010163 -0.00638996  0.00701539  0.00604718 -0.0126149
  0.00158574  0.03772453]


In [3]:
import os, io, requests, numpy as np
from PIL import Image

CAMERA_URL = os.getenv("CAMERA_SERVICE_URL", "http://127.0.0.1:54322")
FRANKY_URL = os.getenv("FRANKY_SERVICE_URL", "http://127.0.0.1:54321")

# camera_service camera_id == str(int(serial))  (leading zeros stripped)
LEFT_CAM  = str(int(os.environ["LEFT_CAMERA_SERIAL"]))
WRIST_CAM = str(int(os.environ["WRIST_CAMERA_SERIAL"]))
# or discover: requests.get(f"{CAMERA_URL}/cameras").json()

def get_rgb(cam_id, q=90):
    r = requests.get(f"{CAMERA_URL}/camera/{cam_id}/rgb.jpg",
                     params={"jpeg_quality": q}, timeout=5)
    r.raise_for_status()
    return np.asarray(Image.open(io.BytesIO(r.content)).convert("RGB"), dtype=np.uint8)

def get_joints():
    r = requests.get(f"{FRANKY_URL}/joint_state", timeout=5)
    r.raise_for_status()
    return np.asarray(r.json()["positions"], dtype=np.float32)  # (7,)


In [4]:
left  = get_rgb(LEFT_CAM)
wrist = get_rgb(WRIST_CAM)
q     = get_joints()
grip  = 0.0   # gripper_service isn't running; fine for a single test
print(left.shape, wrist.shape, q, grip)


(480, 640, 3) (480, 640, 3) [ 5.7782527e-06  1.4503019e-04  3.7766267e-06 -1.5709087e+00
 -6.6348998e-06  1.5709904e+00 -1.0935401e-04] 0.0


In [5]:
from openpi_client import image_tools, websocket_client_policy

client = websocket_client_policy.WebsocketClientPolicy(host="localhost", port=18000)

observation = {
    "observation/exterior_image_1_left": image_tools.resize_with_pad(left, 224, 224),
    "observation/wrist_image_left":      image_tools.resize_with_pad(wrist, 224, 224),
    "observation/joint_position":        q,
    "observation/gripper_position":      np.array([grip], dtype=np.float32),
    "prompt": "pick up the object",
}
actions = np.asarray(client.infer(observation)["actions"])
print(actions.shape, actions[0])   # (15, 8): 7 joint targets + gripper


(15, 8) [ 0.03724137 -0.00454885  0.00783185 -0.01273739  0.00732212  0.01336049
  0.04690729  0.01495225]
